# 10 OpenCV Telea Baseline Report

This notebook generates an interim baseline report for the OpenCV Telea restoration experiment on the controlled 50-painting subset.

The report consolidates:

- dataset and restoration-case overview,
- mask-type and category summaries,
- classical masked-region metrics,
- LPIPS mask-bounding-box metrics,
- CLIP and DINOv2 feature-space metrics,
- metric correlation and disagreement analysis,
- selected diagnostic error-map cases.

The goal is not to claim that OpenCV faithfully restores paintings. OpenCV Telea is used as a deterministic classical baseline to test the evaluation framework before adding pretrained and generative inpainting models.

In [1]:
from pathlib import Path
import sys
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

CONFIG_PATH = PROJECT_ROOT / "config" / "experiment_50_config.yaml"

print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_DIR)
print("Source directory exists:", SRC_DIR.exists())
print("Config path:", CONFIG_PATH)
print("Config exists:", CONFIG_PATH.exists())

if not SRC_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {SRC_DIR}")

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Experiment config not found: {CONFIG_PATH}")

Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Source directory: D:\Masters\FH\Thesis\painting-restoration-eval\src
Source directory exists: True
Config path: D:\Masters\FH\Thesis\painting-restoration-eval\config\experiment_50_config.yaml
Config exists: True


In [2]:
import pandas as pd
import numpy as np
from IPython.display import HTML, display

from restoration_eval.reporting import (
    prepare_opencv_50_report_dataframe,
    summarize_report_overview,
    summarize_report_by_mask_type,
    summarize_report_by_category,
    summarize_metric_correlations,
    select_opencv_50_diagnostic_cases,
    generate_opencv_50_report,
)

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

paths_cfg = config.get("paths", {})

processed_metadata_path = PROJECT_ROOT / paths_cfg["processed_metadata_dir"] / "metadata_processed_clean.csv"
restored_metadata_path = PROJECT_ROOT / paths_cfg["processed_metadata_dir"] / "metadata_restored_opencv_telea.csv"

metrics_dir = PROJECT_ROOT / paths_cfg["metrics_dir"]
reports_dir = PROJECT_ROOT / paths_cfg["reports_dir"]

classical_metrics_path = metrics_dir / "classical_metrics_opencv_telea_50.csv"
lpips_metrics_path = metrics_dir / "lpips_metrics_opencv_telea_50.csv"
feature_metrics_path = metrics_dir / "feature_similarity_opencv_telea_50.csv"
error_map_manifest_path = metrics_dir / "error_map_manifest_all_opencv_telea_50.csv"

report_output_path = reports_dir / "opencv_telea_50_baseline_report.html"
selected_cases_output_path = reports_dir / "opencv_telea_50_report_selected_cases.csv"
report_case_metrics_output_path = reports_dir / "opencv_telea_50_report_case_metrics.csv"
report_summary_mask_output_path = reports_dir / "opencv_telea_50_report_summary_by_mask_type.csv"
report_summary_category_output_path = reports_dir / "opencv_telea_50_report_summary_by_category.csv"
report_correlation_output_path = reports_dir / "opencv_telea_50_report_metric_correlations.csv"

reports_dir.mkdir(parents=True, exist_ok=True)

print("Processed metadata:", processed_metadata_path)
print("Restored metadata:", restored_metadata_path)
print("Classical metrics:", classical_metrics_path)
print("LPIPS metrics:", lpips_metrics_path)
print("Feature metrics:", feature_metrics_path)
print("Error-map manifest:", error_map_manifest_path)
print("Report output:", report_output_path)

Processed metadata: D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata\metadata_processed_clean.csv
Restored metadata: D:\Masters\FH\Thesis\painting-restoration-eval\data\processed\metadata\metadata_restored_opencv_telea.csv
Classical metrics: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\classical_metrics_opencv_telea_50.csv
LPIPS metrics: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\lpips_metrics_opencv_telea_50.csv
Feature metrics: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\feature_similarity_opencv_telea_50.csv
Error-map manifest: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\metrics\error_map_manifest_all_opencv_telea_50.csv
Report output: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_telea_50_baseline_report.html


In [3]:
required_paths = {
    "processed_metadata": processed_metadata_path,
    "restored_metadata": restored_metadata_path,
    "classical_metrics": classical_metrics_path,
    "lpips_metrics": lpips_metrics_path,
    "feature_metrics": feature_metrics_path,
    "error_map_manifest": error_map_manifest_path,
}

missing_paths = {
    name: path
    for name, path in required_paths.items()
    if not path.exists()
}

if missing_paths:
    for name, path in missing_paths.items():
        print(f"Missing {name}: {path}")
    raise FileNotFoundError("One or more required report inputs are missing.")

processed_metadata_df = pd.read_csv(processed_metadata_path)
restored_metadata_df = pd.read_csv(restored_metadata_path)
classical_metrics_df = pd.read_csv(classical_metrics_path)
lpips_metrics_df = pd.read_csv(lpips_metrics_path)
feature_metrics_df = pd.read_csv(feature_metrics_path)
error_map_manifest_df = pd.read_csv(error_map_manifest_path)

print("Loaded input shapes:")
print("processed_metadata_df:", processed_metadata_df.shape)
print("restored_metadata_df:", restored_metadata_df.shape)
print("classical_metrics_df:", classical_metrics_df.shape)
print("lpips_metrics_df:", lpips_metrics_df.shape)
print("feature_metrics_df:", feature_metrics_df.shape)
print("error_map_manifest_df:", error_map_manifest_df.shape)

display(processed_metadata_df.head())
display(restored_metadata_df.head())
display(classical_metrics_df.head())
display(lpips_metrics_df.head())
display(feature_metrics_df.head())
display(error_map_manifest_df.head())

Loaded input shapes:
processed_metadata_df: (50, 43)
restored_metadata_df: (250, 20)
classical_metrics_df: (900, 30)
lpips_metrics_df: (700, 24)
feature_metrics_df: (700, 28)
error_map_manifest_df: (250, 29)


,painting_id,category,title,artist,date,style_or_period,medium,source,source_url,license,...,padding_color_b,content_x_min,content_y_min,content_x_max,content_y_max,content_width,content_height,preprocessing_method,content_area_pixels,content_area_percentage
0,p001,portrait_figure,Juan de Pareja,Diego Velázquez,1650,Baroque,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,...,20,52,0,715,768,663,768,aspect_ratio_resize_median_rgb_pad,509184,86.33
1,p002,portrait_figure,Madame X (Madame Pierre Gautreau),John Singer Sargent,1883-84,19th century portraiture,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,...,35,159,0,608,768,449,768,aspect_ratio_resize_median_rgb_pad,344832,58.46
2,p003,portrait_figure,Marie Joséphine Charlotte du Val d'Ognes,Marie-Denise Villers,1801,Neoclassical / early 19th century,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,...,39,77,0,691,768,614,768,aspect_ratio_resize_median_rgb_pad,471552,79.95
3,p004,portrait_figure,Madame Georges Charpentier and Her Children,Pierre-Auguste Renoir,1878,Impressionism,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,...,62,0,71,768,696,768,625,aspect_ratio_resize_median_rgb_pad,480000,81.38
4,p005,portrait_figure,Self-Portrait with Two Pupils,Adélaïde Labille-Guiard,1785,18th century portraiture,Oil on canvas,The Metropolitan Museum of Art,https://www.metmuseum.org/art/collection/searc...,Public Domain / Open Access,...,34,99,0,668,768,569,768,aspect_ratio_resize_median_rgb_pad,436992,74.09


,case_id,painting_id,mask_id,mask_type,model_name,algorithm,inpaint_radius,clean_filename,clean_path,mask_filename,mask_path,damaged_filename,damaged_path,restored_filename,restored_path,damaged_area_pixels,damaged_area_percentage_content,damaged_area_percentage_full,status,issue
0,p001_loss_large,p001,p001_loss_large,loss_large,opencv_telea,cv2.INPAINT_TELEA,3,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_large_mask.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_large_damaged.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_large_restored_opencv_telea.png,D:\Masters\FH\Thesis\painting-restoration-eval...,67242,13.2058,11.4003,ok,NaN
1,p001_loss_small,p001,p001_loss_small,loss_small,opencv_telea,cv2.INPAINT_TELEA,3,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_small_mask.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_small_damaged.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_small_restored_opencv_telea.png,D:\Masters\FH\Thesis\painting-restoration-eval...,24198,4.7523,4.1026,ok,NaN
2,p001_mixed_damage,p001,p001_mixed_damage,mixed_damage,opencv_telea,cv2.INPAINT_TELEA,3,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_mixed_damage_mask.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_mixed_damage_damaged.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_mixed_damage_restored_opencv_telea.png,D:\Masters\FH\Thesis\painting-restoration-eval...,44392,8.7183,7.5263,ok,NaN
3,p001_scratch_thin,p001,p001_scratch_thin,scratch_thin,opencv_telea,cv2.INPAINT_TELEA,3,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_scratch_thin_mask.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_scratch_thin_damaged.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_scratch_thin_restored_opencv_telea.png,D:\Masters\FH\Thesis\painting-restoration-eval...,8012,1.5735,1.3584,ok,NaN
4,p001_zero_control,p001,p001_zero_control,zero_control,opencv_telea,cv2.INPAINT_TELEA,3,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_zero_control_mask.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_zero_control_damaged.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_zero_control_restored_opencv_telea.png,D:\Masters\FH\Thesis\painting-restoration-eval...,0,0.0000,0.0000,ok,NaN


,case_id,painting_id,category,title,mask_id,mask_type,model_name,evaluation_region,region_pixel_count,region_x_min,...,restored_mae,mae_improvement,damaged_psnr,restored_psnr,psnr_improvement,damaged_ssim,restored_ssim,ssim_improvement,status,issue
0,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,full_image,589824,0.0,...,2.166397,22.892098,10.659855,27.641154,16.981299,0.897178,0.961985,0.064807,ok,NaN
1,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,content_region,509184,52.0,...,2.509491,26.517544,10.021378,27.002677,16.981299,0.880745,0.955910,0.075165,ok,NaN
2,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,masked_region,67242,NaN,...,19.002895,200.801716,1.229037,18.210336,16.981299,NaN,NaN,NaN,ok,NaN
3,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,mask_bbox_crop,126750,221.0,...,10.081204,106.527087,3.982110,20.963409,16.981299,0.512661,0.819825,0.307164,ok,NaN
4,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,opencv_telea,full_image,589824,0.0,...,0.145845,8.718577,15.279670,46.856186,31.576516,0.953800,0.994575,0.040775,ok,NaN


,case_id,painting_id,category,title,mask_id,mask_type,model_name,evaluation_region,region_pixel_count,region_x_min,...,damaged_area_percentage_content,damaged_area_percentage_full,damaged_lpips,restored_lpips,lpips_improvement,lpips_net,crop_resize,device,status,issue
0,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,full_image,589824,0,...,13.2058,11.4003,0.264050,0.088961,0.175089,alex,256,cuda,ok,NaN
1,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,content_region,509184,52,...,13.2058,11.4003,0.288218,0.101456,0.186762,alex,256,cuda,ok,NaN
2,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,mask_bbox_crop,178929,197,...,13.2058,11.4003,0.592107,0.273881,0.318226,alex,256,cuda,ok,NaN
3,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,opencv_telea,full_image,589824,0,...,4.7523,4.1026,0.335288,0.005583,0.329705,alex,256,cuda,ok,NaN
4,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,opencv_telea,content_region,509184,52,...,4.7523,4.1026,0.318926,0.006440,0.312486,alex,256,cuda,ok,NaN


,case_id,painting_id,category,title,mask_id,mask_type,model_name,evaluation_region,region_pixel_count,region_x_min,...,clip_similarity_improvement,dinov2_damaged_similarity,dinov2_restored_similarity,dinov2_similarity_improvement,clip_model_name,dinov2_model_name,feature_resize,device,status,issue
0,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,full_image,589824,0,...,0.035249,0.796219,0.938681,0.142461,openai/clip-vit-base-patch32,dinov2_vits14,224,cuda,ok,NaN
1,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,content_region,509184,52,...,0.047824,0.790295,0.920064,0.129769,openai/clip-vit-base-patch32,dinov2_vits14,224,cuda,ok,NaN
2,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,mask_bbox_crop,178929,197,...,-0.002892,0.706465,0.748291,0.041827,openai/clip-vit-base-patch32,dinov2_vits14,224,cuda,ok,NaN
3,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,opencv_telea,full_image,589824,0,...,0.084412,0.935493,0.998409,0.062916,openai/clip-vit-base-patch32,dinov2_vits14,224,cuda,ok,NaN
4,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,opencv_telea,content_region,509184,52,...,0.078385,0.932966,0.998013,0.065047,openai/clip-vit-base-patch32,dinov2_vits14,224,cuda,ok,NaN


,case_id,painting_id,category,title,mask_id,mask_type,model_name,selection_group,figure_filename,figure_path,...,damaged_error_mean_masked,restored_error_mean_masked,improvement_mean_masked,negative_improvement_pixels_masked,positive_improvement_pixels_masked,zero_improvement_pixels_masked,negative_improvement_percentage_masked,positive_improvement_percentage_masked,status,issue
0,p001_loss_large,p001,portrait_figure,Juan de Pareja,p001_loss_large,loss_large,opencv_telea,all_opencv_cases,p001_loss_large_opencv_telea_error_maps.png,D:\Masters\FH\Thesis\painting-restoration-eval...,...,219.804611,19.002895,200.801697,99,67140,3,0.147229,99.848309,ok,NaN
1,p001_loss_small,p001,portrait_figure,Juan de Pareja,p001_loss_small,loss_small,opencv_telea,all_opencv_cases,p001_loss_small_opencv_telea_error_maps.png,D:\Masters\FH\Thesis\painting-restoration-eval...,...,216.069473,3.554949,212.514511,0,24198,0,0.000000,100.000000,ok,NaN
2,p001_mixed_damage,p001,portrait_figure,Juan de Pareja,p001_mixed_damage,mixed_damage,opencv_telea,all_opencv_cases,p001_mixed_damage_opencv_telea_error_maps.png,D:\Masters\FH\Thesis\painting-restoration-eval...,...,224.467407,7.844874,216.622543,0,44392,0,0.000000,100.000000,ok,NaN
3,p001_scratch_thin,p001,portrait_figure,Juan de Pareja,p001_scratch_thin,scratch_thin,opencv_telea,all_opencv_cases,p001_scratch_thin_opencv_telea_error_maps.png,D:\Masters\FH\Thesis\painting-restoration-eval...,...,217.938065,6.558287,211.379776,0,8012,0,0.000000,100.000000,ok,NaN
4,p001_zero_control,p001,portrait_figure,Juan de Pareja,p001_zero_control,zero_control,opencv_telea,all_opencv_cases,p001_zero_control_opencv_telea_error_maps.png,D:\Masters\FH\Thesis\painting-restoration-eval...,...,NaN,NaN,NaN,0,0,0,NaN,NaN,ok,NaN


In [4]:
expected_counts = {
    "processed paintings": (len(processed_metadata_df), 50),
    "restored cases": (len(restored_metadata_df), 250),
    "classical metric rows": (len(classical_metrics_df), 900),
    "LPIPS metric rows": (len(lpips_metrics_df), 700),
    "feature metric rows": (len(feature_metrics_df), 700),
    "error-map manifest rows": (len(error_map_manifest_df), 250),
}

count_check_rows = []

for label, (actual, expected) in expected_counts.items():
    count_check_rows.append(
        {
            "item": label,
            "actual": actual,
            "expected": expected,
            "passed": actual == expected,
        }
    )

count_check_df = pd.DataFrame(count_check_rows)
display(count_check_df)

if not count_check_df["passed"].all():
    raise ValueError("One or more report input row-count checks failed.")

print("Report input row-count checks passed.")

,item,actual,expected,passed
0,processed paintings,50,50,True
1,restored cases,250,250,True
2,classical metric rows,900,900,True
3,LPIPS metric rows,700,700,True
4,feature metric rows,700,700,True
5,error-map manifest rows,250,250,True


Report input row-count checks passed.


In [5]:
print("Classical metric regions:")
display(classical_metrics_df["evaluation_region"].value_counts().sort_index())

print("\nLPIPS metric regions:")
display(lpips_metrics_df["evaluation_region"].value_counts().sort_index())

print("\nFeature metric regions:")
display(feature_metrics_df["evaluation_region"].value_counts().sort_index())

print("\nRestoration mask types:")
display(restored_metadata_df["mask_type"].value_counts().sort_index())

print("\nPainting categories:")
display(processed_metadata_df["category"].value_counts().sort_index())

# Required local report regions
required_region_checks = [
    (
        "classical masked_region",
        classical_metrics_df[
            (classical_metrics_df["evaluation_region"] == "masked_region")
            & (classical_metrics_df["status"] == "ok")
        ],
        200,
    ),
    (
        "LPIPS mask_bbox_crop",
        lpips_metrics_df[
            (lpips_metrics_df["evaluation_region"] == "mask_bbox_crop")
            & (lpips_metrics_df["status"] == "ok")
        ],
        200,
    ),
    (
        "feature mask_bbox_crop",
        feature_metrics_df[
            (feature_metrics_df["evaluation_region"] == "mask_bbox_crop")
            & (feature_metrics_df["status"] == "ok")
        ],
        200,
    ),
]

region_check_rows = []

for label, df, expected in required_region_checks:
    region_check_rows.append(
        {
            "item": label,
            "actual": len(df),
            "expected": expected,
            "passed": len(df) == expected,
        }
    )

region_check_df = pd.DataFrame(region_check_rows)
display(region_check_df)

if not region_check_df["passed"].all():
    raise ValueError("One or more required metric-region checks failed.")

print("Required report metric-region checks passed.")

Classical metric regions:


evaluation_region
content_region    250
full_image        250
mask_bbox_crop    200
masked_region     200
Name: count, dtype: int64


LPIPS metric regions:


evaluation_region
content_region    250
full_image        250
mask_bbox_crop    200
Name: count, dtype: int64


Feature metric regions:


...

mask_type
loss_large      50
loss_small      50
mixed_damage    50
scratch_thin    50
zero_control    50
Name: count, dtype: int64


Painting categories:


category
abstraction_surrealism     10
architecture_structured    10
high_texture_brushwork     10
landscape_natural          10
portrait_figure            10
Name: count, dtype: int64

,item,actual,expected,passed
0,classical masked_region,200,200,True
1,LPIPS mask_bbox_crop,200,200,True
2,feature mask_bbox_crop,200,200,True


Required report metric-region checks passed.


In [6]:
report_df = prepare_opencv_50_report_dataframe(
    processed_metadata_df=processed_metadata_df,
    restored_metadata_df=restored_metadata_df,
    classical_metrics_df=classical_metrics_df,
    lpips_metrics_df=lpips_metrics_df,
    feature_metrics_df=feature_metrics_df,
    error_map_manifest_df=error_map_manifest_df,
    project_root=PROJECT_ROOT,
    include_zero_control=False,
)

print("Report dataframe shape:", report_df.shape)
print("Expected rows: 200 non-zero mask cases")

display(report_df.head())

print("\nRows by mask type:")
display(report_df["mask_type"].value_counts().sort_index())

print("\nRows by category:")
display(report_df["category"].value_counts().sort_index())

if len(report_df) != 200:
    raise ValueError(f"Expected 200 non-zero report rows, found {len(report_df)}.")

print("Report dataframe built successfully.")

Report dataframe shape: (200, 49)
Expected rows: 200 non-zero mask cases


,case_id,painting_id,mask_id,mask_type,model_name,algorithm,inpaint_radius,clean_filename,clean_path,mask_filename,...,damaged_lpips,restored_lpips,lpips_improvement,clip_damaged_similarity,clip_restored_similarity,clip_similarity_improvement,dinov2_damaged_similarity,dinov2_restored_similarity,dinov2_similarity_improvement,error_map_figure_path
0,p001_loss_large,p001,p001_loss_large,loss_large,opencv_telea,cv2.INPAINT_TELEA,3,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_large_mask.png,...,0.592107,0.273881,0.318226,0.810900,0.808008,-0.002892,0.706465,0.748291,0.041827,D:\Masters\FH\Thesis\painting-restoration-eval...
1,p001_loss_small,p001,p001_loss_small,loss_small,opencv_telea,cv2.INPAINT_TELEA,3,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_loss_small_mask.png,...,0.341009,0.007826,0.333183,0.910694,0.999298,0.088604,0.948942,0.997888,0.048946,D:\Masters\FH\Thesis\painting-restoration-eval...
2,p001_mixed_damage,p001,p001_mixed_damage,mixed_damage,opencv_telea,cv2.INPAINT_TELEA,3,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_mixed_damage_mask.png,...,0.416328,0.030484,0.385844,0.895807,0.994364,0.098557,0.795198,0.974854,0.179656,D:\Masters\FH\Thesis\painting-restoration-eval...
3,p001_scratch_thin,p001,p001_scratch_thin,scratch_thin,opencv_telea,cv2.INPAINT_TELEA,3,p001_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p001_scratch_thin_mask.png,...,0.425347,0.005461,0.419886,0.857966,0.999632,0.141666,0.924942,0.998968,0.074026,D:\Masters\FH\Thesis\painting-restoration-eval...
4,p002_loss_large,p002,p002_loss_large,loss_large,opencv_telea,cv2.INPAINT_TELEA,3,p002_clean.png,D:\Masters\FH\Thesis\painting-restoration-eval...,p002_loss_large_mask.png,...,0.589203,0.202087,0.387116,0.724333,0.823848,0.099515,0.762878,0.774365,0.011488,D:\Masters\FH\Thesis\painting-restoration-eval...



Rows by mask type:


mask_type
loss_large      50
loss_small      50
mixed_damage    50
scratch_thin    50
Name: count, dtype: int64


Rows by category:


category
abstraction_surrealism     40
architecture_structured    40
high_texture_brushwork     40
landscape_natural          40
portrait_figure            40
Name: count, dtype: int64

Report dataframe built successfully.


In [7]:
required_report_columns = [
    "case_id",
    "painting_id",
    "category",
    "title",
    "mask_type",
    "model_name",
    "mse_improvement",
    "mae_improvement",
    "psnr_improvement",
    "lpips_improvement",
    "clip_similarity_improvement",
    "dinov2_similarity_improvement",
    "error_map_figure_path",
]

missing_report_columns = [
    column for column in required_report_columns
    if column not in report_df.columns
]

if missing_report_columns:
    raise ValueError(f"Missing report columns: {missing_report_columns}")

null_check_columns = [
    "case_id",
    "painting_id",
    "category",
    "title",
    "mask_type",
    "mse_improvement",
    "lpips_improvement",
    "clip_similarity_improvement",
    "dinov2_similarity_improvement",
]

null_summary_df = (
    report_df[null_check_columns]
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_values"})
)

display(null_summary_df)

if (null_summary_df["missing_values"] > 0).any():
    raise ValueError("Some required report metric columns contain missing values.")

print("Required report columns are complete.")

,column,missing_values
0,case_id,0
1,painting_id,0
2,category,0
3,title,0
4,mask_type,0
5,mse_improvement,0
6,lpips_improvement,0
7,clip_similarity_improvement,0
8,dinov2_similarity_improvement,0


Required report columns are complete.


In [8]:
# Validate selected error-map figure paths before report generation

figure_path_check_df = report_df[
    ["case_id", "painting_id", "mask_type", "error_map_figure_path"]
].copy()

figure_path_check_df["has_figure_path"] = (
    figure_path_check_df["error_map_figure_path"]
    .fillna("")
    .astype(str)
    .str.len()
    > 0
)

figure_path_check_df["figure_exists"] = figure_path_check_df["error_map_figure_path"].map(
    lambda value: Path(str(value)).exists() if pd.notna(value) and str(value).strip() else False
)

print("Figure path availability:")
display(
    figure_path_check_df[
        ["has_figure_path", "figure_exists"]
    ].value_counts().reset_index(name="cases")
)

missing_figure_rows = figure_path_check_df[~figure_path_check_df["figure_exists"]]

if not missing_figure_rows.empty:
    print("Rows with missing error-map figures:")
    display(missing_figure_rows.head(20))
    raise FileNotFoundError("Some report error-map figure paths are missing.")

print("All report error-map figure paths exist.")

Figure path availability:


,has_figure_path,figure_exists,cases
0,True,True,200


All report error-map figure paths exist.


In [9]:
report_df.to_csv(report_case_metrics_output_path, index=False)

print("Saved report case metrics:")
print(report_case_metrics_output_path)
print("Exists:", report_case_metrics_output_path.exists())
print("Rows:", len(report_df))

Saved report case metrics:
D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_telea_50_report_case_metrics.csv
Exists: True
Rows: 200


In [10]:
overview_df = summarize_report_overview(
    processed_metadata_df=processed_metadata_df,
    restored_metadata_df=restored_metadata_df,
    report_df=report_df,
)

summary_by_mask_df = summarize_report_by_mask_type(report_df)
summary_by_category_df = summarize_report_by_category(report_df)
correlation_df = summarize_metric_correlations(report_df)

print("Experiment overview:")
display(overview_df)

print("\nSummary by mask type:")
display(summary_by_mask_df)

print("\nSummary by category:")
display(summary_by_category_df)

print("\nMain metric correlation matrix:")
display(correlation_df)

Experiment overview:


,item,value
0,Paintings,50
1,Painting categories,5
2,Restoration cases generated,250
3,Non-zero report cases,200
4,Mask types,5
5,Baseline model,opencv_telea



Summary by mask type:


,mask_type,cases,mean_damage_area_content_pct,mean_mse_improvement,mean_restored_mse,mean_lpips_improvement,mean_restored_lpips,mean_clip_improvement,clip_improvement_rate,mean_dinov2_improvement,dinov2_improvement_rate
0,loss_large,50,12.82220,23999.73331,1580.98980,0.18588,0.29941,0.02631,0.74,-0.12213,0.16
1,loss_small,50,4.43850,25902.88156,616.99864,0.16653,0.02622,0.09881,1.00,0.05793,1.00
2,mixed_damage,50,10.13509,26381.81926,855.77489,0.26445,0.04943,0.13240,1.00,0.17280,1.00
3,scratch_thin,50,2.19846,28319.66895,325.06396,0.25659,0.00363,0.10116,1.00,0.14765,1.00



Summary by category:


,category,cases,mean_mse_improvement,mean_lpips_improvement,mean_clip_improvement,mean_dinov2_improvement,dinov2_negative_rate
0,abstraction_surrealism,40,18899.95990,0.13913,0.06509,0.02661,0.250
1,architecture_structured,40,25406.39262,0.23882,0.11136,0.03398,0.250
2,high_texture_brushwork,40,26778.38258,0.23651,0.09135,0.09313,0.175
3,landscape_natural,40,21573.90409,0.20719,0.09035,0.08718,0.250
4,portrait_figure,40,38096.48966,0.27016,0.09020,0.07941,0.125



Main metric correlation matrix:


,mse_improvement,mae_improvement,psnr_improvement,lpips_improvement,clip_similarity_improvement,dinov2_similarity_improvement
mse_improvement,1.0000,0.9789,0.6746,0.5740,0.2246,0.1861
mae_improvement,0.9789,1.0000,0.7646,0.6124,0.2881,0.2507
psnr_improvement,0.6746,0.7646,1.0000,0.6052,0.3936,0.4010
lpips_improvement,0.5740,0.6124,0.6052,1.0000,0.5126,0.5023
clip_similarity_improvement,0.2246,0.2881,0.3936,0.5126,1.0000,0.6151
dinov2_similarity_improvement,0.1861,0.2507,0.4010,0.5023,0.6151,1.0000


In [11]:
summary_by_mask_df.to_csv(report_summary_mask_output_path, index=False)
summary_by_category_df.to_csv(report_summary_category_output_path, index=False)
correlation_df.to_csv(report_correlation_output_path)

overview_output_path = reports_dir / "opencv_telea_50_report_overview.csv"
overview_df.to_csv(overview_output_path, index=False)

print("Saved overview:", overview_output_path)
print("Exists:", overview_output_path.exists())

print("\nSaved summary by mask type:", report_summary_mask_output_path)
print("Exists:", report_summary_mask_output_path.exists())

print("\nSaved summary by category:", report_summary_category_output_path)
print("Exists:", report_summary_category_output_path.exists())

print("\nSaved metric correlations:", report_correlation_output_path)
print("Exists:", report_correlation_output_path.exists())

Saved overview: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_telea_50_report_overview.csv
Exists: True

Saved summary by mask type: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_telea_50_report_summary_by_mask_type.csv
Exists: True

Saved summary by category: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_telea_50_report_summary_by_category.csv
Exists: True

Saved metric correlations: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_telea_50_report_metric_correlations.csv
Exists: True


In [12]:
selected_cases_df = select_opencv_50_diagnostic_cases(
    report_df,
    n_per_group=3,
    include_category_examples=True,
)

selected_cases_df.to_csv(selected_cases_output_path, index=False)

print("Selected diagnostic cases:", len(selected_cases_df))
print("Saved selected cases:", selected_cases_output_path)
print("Exists:", selected_cases_output_path.exists())

display(
    selected_cases_df[
        [
            "case_id",
            "painting_id",
            "category",
            "title",
            "mask_type",
            "selection_reason",
            "mse_improvement",
            "lpips_improvement",
            "clip_similarity_improvement",
            "dinov2_similarity_improvement",
        ]
    ]
)

Selected diagnostic cases: 20
Saved selected cases: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_telea_50_report_selected_cases.csv
Exists: True


,case_id,painting_id,category,title,mask_type,selection_reason,mse_improvement,lpips_improvement,clip_similarity_improvement,dinov2_similarity_improvement
0,p001_scratch_thin,p001,portrait_figure,Juan de Pareja,scratch_thin,Strongest mask-bbox LPIPS improvement,48335.436569,0.419886,0.141666,0.074026
1,p006_loss_large,p006,portrait_figure,Boy with a Sword,loss_large,Strongest masked-region MSE improvement,51838.575409,0.383844,0.016472,0.016074
2,p006_mixed_damage,p006,portrait_figure,Boy with a Sword,mixed_damage,Strong category example by MSE improvement; St...,51841.337265,0.408723,0.156120,0.211854
3,p006_scratch_thin,p006,portrait_figure,Boy with a Sword,scratch_thin,Strongest mask-bbox LPIPS improvement; Stronge...,50997.089211,0.461126,0.126735,0.187726
4,p009_loss_large,p009,portrait_figure,Madame Roulin and Her Baby,loss_large,Weakest DINOv2 feature improvement,18937.032227,0.277251,-0.066163,-0.299740
5,p010_mixed_damage,p010,portrait_figure,Joan of Arc,mixed_damage,Strongest mask-bbox LPIPS improvement,39740.788147,0.440936,0.179592,0.299598
6,p014_loss_large,p014,landscape_natural,Landscape on a River,loss_large,Weakest DINOv2 feature improvement,26803.850098,0.223294,-0.011446,-0.304017
7,p014_mixed_damage,p014,landscape_natural,Landscape on a River,mixed_damage,Strongest DINOv2 feature improvement,24221.775665,0.372342,0.160194,0.468329
8,p017_loss_large,p017,landscape_natural,Picturesque Landscape,loss_large,Weakest mask-bbox LPIPS improvement,8919.444336,0.032902,-0.030936,-0.286992
9,p019_loss_small,p019,landscape_natural,Evening: Landscape with an Aqueduct,loss_small,Strong category example by MSE improvement,44883.554977,0.127734,0.088949,0.015290


In [13]:
print("Selected cases by mask type:")
display(selected_cases_df["mask_type"].value_counts().sort_index())

print("\nSelected cases by category:")
display(selected_cases_df["category"].value_counts().sort_index())

print("\nSelected cases by selection reason:")
selection_reason_counts = (
    selected_cases_df["selection_reason"]
    .str.split("; ")
    .explode()
    .value_counts()
    .reset_index()
)

selection_reason_counts.columns = ["selection_reason", "cases"]
display(selection_reason_counts)

print("\nSelected case figure path check:")
selected_cases_figure_check_df = selected_cases_df[
    ["case_id", "error_map_figure_path"]
].copy()

selected_cases_figure_check_df["figure_exists"] = selected_cases_figure_check_df[
    "error_map_figure_path"
].map(lambda value: Path(str(value)).exists() if pd.notna(value) and str(value).strip() else False)

display(selected_cases_figure_check_df["figure_exists"].value_counts().reset_index(name="cases"))

if not selected_cases_figure_check_df["figure_exists"].all():
    display(selected_cases_figure_check_df[~selected_cases_figure_check_df["figure_exists"]])
    raise FileNotFoundError("One or more selected case figures are missing.")

print("Selected case coverage and figure checks passed.")

Selected cases by mask type:


mask_type
loss_large      7
loss_small      3
mixed_damage    6
scratch_thin    4
Name: count, dtype: int64


Selected cases by category:


category
abstraction_surrealism     5
architecture_structured    1
high_texture_brushwork     3
landscape_natural          5
portrait_figure            6
Name: count, dtype: int64


Selected cases by selection reason:


,selection_reason,cases
0,Strong category example by MSE improvement,5
1,Strongest mask-bbox LPIPS improvement,3
2,Strongest masked-region MSE improvement,3
3,Weakest DINOv2 feature improvement,3
4,Strongest DINOv2 feature improvement,3
5,Weakest mask-bbox LPIPS improvement,3
6,Weakest masked-region MSE improvement,3



Selected case figure path check:


,figure_exists,cases
0,True,20


Selected case coverage and figure checks passed.


In [17]:
report_outputs = generate_opencv_50_report(
    processed_metadata_df=processed_metadata_df,
    restored_metadata_df=restored_metadata_df,
    classical_metrics_df=classical_metrics_df,
    lpips_metrics_df=lpips_metrics_df,
    feature_metrics_df=feature_metrics_df,
    error_map_manifest_df=error_map_manifest_df,
    project_root=PROJECT_ROOT,
    output_path=report_output_path,
    selected_cases_output_path=selected_cases_output_path,
    image_mode="embedded",
)

print("Report generated.")
print("HTML report:", report_output_path)
print("Exists:", report_output_path.exists())
print("Size MB:", report_output_path.stat().st_size / (1024 * 1024))

print("\nReturned output keys:")
print(list(report_outputs.keys()))

Report generated.
HTML report: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\reports\opencv_telea_50_baseline_report.html
Exists: True
Size MB: 36.21888065338135

Returned output keys:
['report_df', 'overview_df', 'summary_by_mask_df', 'summary_by_category_df', 'correlation_df', 'selected_cases_df', 'html_report']


In [18]:
generated_report_df = report_outputs["report_df"]
generated_overview_df = report_outputs["overview_df"]
generated_summary_by_mask_df = report_outputs["summary_by_mask_df"]
generated_summary_by_category_df = report_outputs["summary_by_category_df"]
generated_correlation_df = report_outputs["correlation_df"]
generated_selected_cases_df = report_outputs["selected_cases_df"]

print("Generated report dataframe:", generated_report_df.shape)
print("Generated overview:", generated_overview_df.shape)
print("Generated summary by mask:", generated_summary_by_mask_df.shape)
print("Generated summary by category:", generated_summary_by_category_df.shape)
print("Generated correlation matrix:", generated_correlation_df.shape)
print("Generated selected cases:", generated_selected_cases_df.shape)

print("\nGenerated selected cases:")
display(
    generated_selected_cases_df[
        [
            "case_id",
            "painting_id",
            "category",
            "mask_type",
            "selection_reason",
            "mse_improvement",
            "lpips_improvement",
            "clip_similarity_improvement",
            "dinov2_similarity_improvement",
        ]
    ]
)

Generated report dataframe: (200, 49)
Generated overview: (6, 2)
Generated summary by mask: (4, 11)
Generated summary by category: (5, 7)
Generated correlation matrix: (6, 6)
Generated selected cases: (20, 51)

Generated selected cases:


,case_id,painting_id,category,mask_type,selection_reason,mse_improvement,lpips_improvement,clip_similarity_improvement,dinov2_similarity_improvement
0,p001_scratch_thin,p001,portrait_figure,scratch_thin,Strongest mask-bbox LPIPS improvement,48335.436569,0.419886,0.141666,0.074026
1,p006_loss_large,p006,portrait_figure,loss_large,Strongest masked-region MSE improvement,51838.575409,0.383844,0.016472,0.016074
2,p006_mixed_damage,p006,portrait_figure,mixed_damage,Strong category example by MSE improvement; St...,51841.337265,0.408723,0.156120,0.211854
3,p006_scratch_thin,p006,portrait_figure,scratch_thin,Strongest mask-bbox LPIPS improvement; Stronge...,50997.089211,0.461126,0.126735,0.187726
4,p009_loss_large,p009,portrait_figure,loss_large,Weakest DINOv2 feature improvement,18937.032227,0.277251,-0.066163,-0.299740
5,p010_mixed_damage,p010,portrait_figure,mixed_damage,Strongest mask-bbox LPIPS improvement,39740.788147,0.440936,0.179592,0.299598
6,p014_loss_large,p014,landscape_natural,loss_large,Weakest DINOv2 feature improvement,26803.850098,0.223294,-0.011446,-0.304017
7,p014_mixed_damage,p014,landscape_natural,mixed_damage,Strongest DINOv2 feature improvement,24221.775665,0.372342,0.160194,0.468329
8,p017_loss_large,p017,landscape_natural,loss_large,Weakest mask-bbox LPIPS improvement,8919.444336,0.032902,-0.030936,-0.286992
9,p019_loss_small,p019,landscape_natural,loss_small,Strong category example by MSE improvement,44883.554977,0.127734,0.088949,0.015290


In [19]:
relative_report_path = report_output_path.relative_to(PROJECT_ROOT)

display(
    HTML(
        f"""
        <p>
            Open the generated report here:
            <a href="../{relative_report_path.as_posix()}" target="_blank">
                {relative_report_path.as_posix()}
            </a>
        </p>
        """
    )
)

## OpenCV baseline report result

The OpenCV Telea baseline report was generated for the controlled 50-painting subset.

The report consolidates:

- experiment overview,
- non-zero restoration case metrics,
- summary by mask type,
- summary by painting category,
- metric correlation matrix,
- selected diagnostic cases,
- linked diagnostic error-map figures.

The report embeds the selected diagnostic figures directly into the HTML file. Since only selected cases are included, this keeps the report portable without generating an excessively large file.

The selected diagnostic cases are not intended to provide equal category coverage. They are selected to expose strong and weak metric behavior, feature-space disagreement, and representative category examples. The full metric CSV files remain the complete quantitative record.

The main baseline finding is that OpenCV Telea reliably improves over white-filled synthetic damage, especially for local and scratch-like damage. However, the baseline remains limited for large missing regions and cases requiring structural or semantic reconstruction.

The disagreement between classical metrics, LPIPS, CLIP, DINOv2, and visual error maps supports the central thesis argument: trustworthy evaluation of AI-assisted painting restoration requires multiple complementary metrics and visual diagnostics rather than a single scalar score.